<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Python_Multi_Turn_Prompting_Examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import json
import time
import random

# IMPORTANT: The API key is automatically provided in the Canvas environment.
# Do not modify this line.
API_KEY = ""
MODEL_ID = "gemini-2.5-flash-preview-05-20"
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL_ID}:generateContent?key={API_KEY}"

def generate_response(chat_history):
    """
    Sends a chat history to the Gemini API and returns the text response.
    Includes exponential backoff for robust API calls.
    """
    retries = 5
    delay = 1
    for i in range(retries):
        try:
            payload = {"contents": chat_history}
            response = requests.post(API_URL, json=payload)
            response.raise_for_status()  # Raise an exception for bad status codes

            result = response.json()
            if result.get("candidates") and len(result["candidates"]) > 0 and \
               result["candidates"][0].get("content") and \
               result["candidates"][0]["content"].get("parts") and \
               len(result["candidates"][0]["content"]["parts"]) > 0:
                text = result["candidates"][0]["content"]["parts"][0]["text"]
                return text
            else:
                print(f"Warning: Unexpected API response structure on attempt {i+1}.")
                print(f"Response content: {json.dumps(result, indent=2)}")
                raise ValueError("Unexpected API response structure.")

        except requests.exceptions.RequestException as e:
            print(f"API call failed on attempt {i+1}: {e}")
            if i < retries - 1:
                sleep_time = delay * (2 ** i) + random.uniform(0, 1)
                print(f"Retrying in {sleep_time:.2f} seconds...")
                time.sleep(sleep_time)
            else:
                print("All retries failed. Giving up.")
                return "API ERROR: Failed to get response after multiple retries."
        except ValueError as e:
            print(f"Error parsing API response on attempt {i+1}: {e}")
            return "API ERROR: Failed to parse response."
    return None

def main():
    """
    Demonstrates multi-turn prompting in three different scenarios.
    """

    print("--- Scenario 1: Basic Multi-Turn Dialogue ---")
    print("Goal: Plan a simple trip to Paris.")

    # Initialize chat history with the system prompt
    chat_history_1 = [
        {"role": "user", "parts": [{"text": "You are a travel agent. I need help planning a 3-day trip to Paris for two people. Start by asking for my travel dates and budget."}]}
    ]

    # Turn 1: Initial user query
    print(f"\nUser: {chat_history_1[-1]['parts'][0]['text']}")
    response_1 = generate_response(chat_history_1)
    chat_history_1.append({"role": "model", "parts": [{"text": response_1}]})
    print(f"Model: {response_1}")

    # Turn 2: User provides more details
    user_query_2 = "The dates are July 20-22, and our total budget is $1,500. Can you suggest a hotel and some activities?"
    chat_history_1.append({"role": "user", "parts": [{"text": user_query_2}]})
    print(f"\nUser: {user_query_2}")
    response_2 = generate_response(chat_history_1)
    chat_history_1.append({"role": "model", "parts": [{"text": response_2}]})
    print(f"Model: {response_2}")

    print("\n-----------------------------------------------------")
    print("--- Scenario 2: Multi-Turn with Persona & Roles ---")
    print("Goal: Simulate a conversation between a skeptical customer and a friendly customer service agent.")

    # Initialize chat history with both roles defined
    chat_history_2 = [
        {"role": "user", "parts": [{"text": "You are a friendly customer service agent. The user is a skeptical customer who is unhappy about a recent purchase. Your goal is to patiently address their concerns and resolve the issue. The user says: 'I bought this product last week and it's already broken. I'm not happy about this at all.' "}]
        }
    ]

    # Turn 1: User's initial complaint (part of the prompt)
    print(f"\nUser: {chat_history_2[-1]['parts'][0]['text'].split('The user says: ')[-1]}")
    response_1_b = generate_response(chat_history_2)
    chat_history_2.append({"role": "model", "parts": [{"text": response_1_b}]})
    print(f"Model: {response_1_b}")

    # Turn 2: Customer continues with more skepticism
    user_query_2_b = "I doubt you can help. Last time this happened, I was on the phone for an hour."
    chat_history_2.append({"role": "user", "parts": [{"text": f"The user says: '{user_query_2_b}'"}]})
    print(f"\nUser: {user_query_2_b}")
    response_2_b = generate_response(chat_history_2)
    chat_history_2.append({"role": "model", "parts": [{"text": response_2_b}]})
    print(f"Model: {response_2_b}")

    print("\n-----------------------------------------------------")
    print("--- Scenario 3: Context Summarization to Prevent 'Answer Bloat' ---")
    print("Goal: Manage a long conversation by periodically summarizing context.")

    # A long, multi-turn conversation
    long_chat = [
        {"role": "user", "parts": [{"text": "Summarize the following article: 'The global economy is facing a series of interconnected challenges, including persistent inflation, supply chain disruptions, and rising geopolitical tensions. Central banks are struggling to balance controlling inflation with avoiding a recession, leading to a period of significant economic uncertainty.'"}]},
        {"role": "model", "parts": [{"text": "The global economy is facing multiple issues like inflation and supply chain problems. Central banks are trying to manage inflation without causing a recession, leading to uncertainty."}]},
        {"role": "user", "parts": [{"text": "Now, explain the concept of 'stagflation' and whether it applies to the current situation described."}]},
        {"role": "model", "parts": [{"text": "Stagflation is a situation with high inflation, high unemployment, and stagnant demand. While not fully in a state of stagflation, the current situation has some similar characteristics, particularly persistent inflation combined with slowing growth."}]},
        {"role": "user", "parts": [{"text": "What is the role of central banks in this?"}]}
    ]

    # Turn 1: Final question without a summary
    print("\nExample without summarization:")
    response_without_summary = generate_response(long_chat)
    print(f"Final response (without summary): {response_without_summary}")

    # Now, let's create a new prompt with a summary
    # This is a key technique to combat "context loss" from the Canvas
    summary_prompt = "Summarize the key points of our conversation so far in one sentence."
    summary_response = generate_response(long_chat + [{"role": "user", "parts": [{"text": summary_prompt}]}] )

    # Create a new, clean chat history starting with the summary
    summarized_chat = [
        {"role": "user", "parts": [{"text": f"Based on our previous conversation, here is a summary: '{summary_response}'. Now, what is the role of central banks?"}]}
    ]

    print("\nExample with summarization:")
    response_with_summary = generate_response(summarized_chat)
    print(f"Final response (with summary): {response_with_summary}")


if __name__ == "__main__":
    main()